# Harris County, TX — Overture buildings (Phase 1, step 1)

First link in the local DuckDB chain. Goal: pull every Overture building inside the
Harris County bbox into a local parquet, confirm the count, render a sample on a
folium map. No GCP. No hazard yet. No scoring yet.

Subsequent notebooks add NFHL flood zones, 3DEP elevation sampling, depth-damage
scoring, and EAD integration on top of the parquet this notebook produces.

**Inputs:** Overture S3 (anonymous read).  
**Output:** `data/raw/harris_buildings.parquet` (gitignored).


In [ ]:
from pathlib import Path
import duckdb
import folium
import json


In [ ]:
# Pinned Overture release. Public releases are retained ~60 days; bump this when
# the upstream rotates. Latest at time of writing: 2026-04-15.0.
OVERTURE_RELEASE = "2026-04-15.0"
OVERTURE_BUILDINGS = (
    f"s3://overturemaps-us-west-2/release/{OVERTURE_RELEASE}"
    "/theme=buildings/type=building/*"
)

# Harris County, TX bounding box. Rough envelope; we'll clip to the actual
# county polygon in a later notebook when we add the Census TIGER boundary.
MIN_LON, MAX_LON = -95.91, -94.90
MIN_LAT, MAX_LAT = 29.49, 30.18

REPO_ROOT = Path.cwd().resolve().parents[1]
OUT_PATH = REPO_ROOT / "data" / "raw" / "harris_buildings.parquet"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUT_PATH


## DuckDB setup

`spatial` for geometry types and WKB IO, `httpfs` for direct S3 reads. Overture
lives in `us-west-2` and is anonymously readable, so we set the region and skip
credentials entirely.


In [ ]:
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='us-west-2';")
con.execute("SET enable_progress_bar=true;")
con.execute("PRAGMA threads=8;")
con.execute("SELECT version()").fetchone()


## Pull buildings inside the Harris County bbox

Overture's parquet schema includes a `bbox` struct on every row, so the bbox
filter pushes down to row-group pruning — no need to read full geometry for
buildings outside the envelope. The `<=` / `>=` form catches any building whose
envelope overlaps ours (vs. a strict-inside filter that would miss edge cases).

Skipping if the local parquet already exists; delete the file to re-pull.


In [ ]:
if OUT_PATH.exists():
    print(f"already have {OUT_PATH}; skipping fetch")
else:
    con.execute(f"""
        COPY (
            SELECT
                id,
                names.primary AS name,
                class,
                subtype,
                num_floors,
                height,
                bbox,
                geometry
            FROM read_parquet('{OVERTURE_BUILDINGS}', filename=false, hive_partitioning=1)
            WHERE bbox.xmin <= {MAX_LON}
              AND bbox.xmax >= {MIN_LON}
              AND bbox.ymin <= {MAX_LAT}
              AND bbox.ymax >= {MIN_LAT}
        )
        TO '{OUT_PATH}' (FORMAT PARQUET, COMPRESSION ZSTD);
    """)
    print(f"wrote {OUT_PATH}")


In [ ]:
summary = con.execute(f"""
    SELECT
        COUNT(*) AS n_buildings,
        COUNT(class) AS n_with_class,
        COUNT(num_floors) AS n_with_floors,
        COUNT(height) AS n_with_height
    FROM read_parquet('{OUT_PATH}')
""").fetchdf()
summary


In [ ]:
con.execute(f"""
    SELECT class, COUNT(*) AS n
    FROM read_parquet('{OUT_PATH}')
    GROUP BY class
    ORDER BY n DESC
    LIMIT 15
""").fetchdf()


## Eyeball check: 2000 buildings on a folium map

Pulling the full county would crush the browser. A 2000-row sample is enough to
confirm the geometries land in the right place and the bbox filter caught the
right region.


In [ ]:
sample = con.execute(f"""
    SELECT
        id,
        class,
        ST_AsGeoJSON(geometry) AS geojson
    FROM read_parquet('{OUT_PATH}')
    USING SAMPLE 2000
""").fetchall()

center_lat = (MIN_LAT + MAX_LAT) / 2
center_lon = (MIN_LON + MAX_LON) / 2
m = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles="cartodbpositron")

for _id, _class, geojson in sample:
    folium.GeoJson(
        json.loads(geojson),
        style_function=lambda _f: {"color": "#1f77b4", "weight": 0.5, "fillOpacity": 0.4},
        tooltip=f"{_class or 'unclassified'}",
    ).add_to(m)

m


## Next

1. Pull FEMA NFHL for Texas, filter to Harris County, write `data/raw/harris_nfhl.parquet`.
2. Pull a 3DEP DEM tile mosaic over the county bbox; sample elevation at building
   centroid.
3. Compute depth at 100-yr and 500-yr from `BFE - elevation`, apply HAZUS DDFs by
   archetype, integrate to EAD.
4. Render an EAD choropleth on folium.
